# Tag SVD content recommender — pipeline validation

**Goal of this notebook** (examiner + team comparison)

1. Validate the Tag SVD content model (Stage 1 model #5) end-to-end on warm / cold / validation tracks.
2. Report numbers in the same format as `sentence_bert_smoke.ipynb` so cross-model comparison is just two notebooks side-by-side.
3. Document the headline negative finding: structured tags + nutrition are **much weaker** content signal than free-form recipe text.

**What this model does** (defined in `src/models/tag_svd_content.py`)

- Recipe representation: the 107-dim feature matrix from `src/data/features.py` (100-dim tag SVD + 7-dim normalized nutrition), already cached at `data/processed/recipe_features.parquet`.
- User profile: L2-normalized mean of positive-recipe vectors (same logic as the SBERT model).
- Ranking: cosine similarity.
- Cold users fall back to popularity.

**Same Stage 1 contract as all other models**: `.fit(train_df) -> self` + `.recommend(user_id, k, exclude_seen=True) -> list[int]`.

**Roadmap**

1. Load training data
2. Inspect the feature representation
3. Fit Tag SVD
4. Sanity check — what does a user's top-5 look like?
5. Eval across all three tracks (validation / warm / cold)
6. Comparison vs SBERT + Popularity
7. Summary

## Setup

In [1]:
import os
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

def _ensure_project_root():
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            os.chdir(candidate)
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not locate the PantryPlate project root.")

PROJECT_ROOT = _ensure_project_root()
print(f"Project root: {PROJECT_ROOT}")

from src.data.loader import load_train_interactions, load_recipes, time_based_split
from src.data.features import build_recipe_feature_matrix
from src.models.tag_svd_content import TagSVDRecommender
from src.models.popularity import PopularityRecommender
from src.eval.harness import evaluate

@contextmanager
def timer(label):
    t0 = time.time()
    yield
    print(f"  [{time.time() - t0:.1f}s] {label}")

Project root: /Users/ikhyvicky/Documents/MITB_stuff/CS608Project2


## 1. Load training data

Same protocol as the SBERT notebook: load the authors' pre-split train, then `time_based_split` for the warm-track LOO holdout.

In [2]:
with timer("loaded train interactions"):
    full_train = load_train_interactions()
with timer("time-based LOO split"):
    train, warm_holdout = time_based_split(full_train, holdout_per_user=1)

print(f"Full train rows: {len(full_train):>10,}")
print(f"  train (post-split): {len(train):>10,}")
print(f"  warm holdout:       {len(warm_holdout):>10,}")
print(f"Unique users in train: {train['user_id'].nunique():,}")

  [0.3s] loaded train interactions


  [0.2s] time-based LOO split
Full train rows:    681,944
  train (post-split):    657,562
  warm holdout:           24,382
Unique users in train: 24,961


## 2. Inspect the feature representation

Each recipe is a 107-dim vector: 100 tag-SVD dimensions (compressed co-occurrence of ~140 selected tags) + 7 normalized nutrition dimensions (calories + 6 PDV percentages, robust-scaled with 99th percentile clipping).

All features are pre-built and cached by `src/data/features.py`.

In [3]:
with timer("loaded feature matrix from cache"):
    features = build_recipe_feature_matrix()

print(f"Feature matrix shape: {features.shape}")
print(f"Index name: {features.index.name}")
print(f"Column groups:")
tag_cols = [c for c in features.columns if c.startswith('tag_svd_')]
nut_cols = [c for c in features.columns if c.startswith('nutrition_')]
print(f"  Tag SVD:    {len(tag_cols)} dims  ({tag_cols[0]} ... {tag_cols[-1]})")
print(f"  Nutrition:  {len(nut_cols)} dims  ({nut_cols})")
print()
print("Sample row (first 5 tag dims + all nutrition):")
print(features.iloc[0, :5].to_dict())
print(features.iloc[0, -7:].to_dict())

  [0.5s] loaded feature matrix from cache
Feature matrix shape: (231637, 107)
Index name: recipe_id
Column groups:
  Tag SVD:    100 dims  (tag_svd_0 ... tag_svd_99)
  Nutrition:  7 dims  (['nutrition_calories', 'nutrition_total_fat_pdv', 'nutrition_sugar_pdv', 'nutrition_sodium_pdv', 'nutrition_protein_pdv', 'nutrition_sat_fat_pdv', 'nutrition_carbs_pdv'])

Sample row (first 5 tag dims + all nutrition):
{'tag_svd_0': 0.43457111606581184, 'tag_svd_1': 0.07839040598354295, 'tag_svd_2': -0.07641129345788346, 'tag_svd_3': 0.13719742184974615, 'tag_svd_4': 0.12524505197697156}
{'nutrition_calories': -0.7584708948740224, 'nutrition_total_fat_pdv': -0.6060606060606061, 'nutrition_sugar_pdv': -0.2033898305084746, 'nutrition_sodium_pdv': -0.5, 'nutrition_protein_pdv': -0.36363636363636365, 'nutrition_sat_fat_pdv': -0.5111111111111111, 'nutrition_carbs_pdv': -0.4166666666666667}


## 3. Fit Tag SVD

We need two fits because the eval setup differs by track:

- `m_full` fit on `full_train` for **validation + cold** evaluation (validation/test items are not in train, so no leakage from using the full data)
- `m_part` fit on `train` (post-LOO split) for **warm** evaluation (held-out warm items must NOT be in the train passed to the model)

Fit is fast (~2s each) because the feature matrix is pre-cached.

In [4]:
with timer("Tag SVD fit on full_train"):
    m_full = TagSVDRecommender()
    m_full.fit(full_train)

with timer("Tag SVD fit on train_part"):
    m_part = TagSVDRecommender()
    m_part.fit(train)

with timer("Popularity baseline fit"):
    pop = PopularityRecommender().fit(train)

print(f"\nm_full: recipe matrix {m_full._recipe_matrix.shape}, user profiles {len(m_full._user_vectors):,}")
print(f"m_part: recipe matrix {m_part._recipe_matrix.shape}, user profiles {len(m_part._user_vectors):,}")

  [2.2s] Tag SVD fit on full_train


  [2.0s] Tag SVD fit on train_part


  [0.3s] Popularity baseline fit

m_full: recipe matrix (231637, 107), user profiles 24,846
m_part: recipe matrix (231637, 107), user profiles 24,225


## 4. Sanity check — what does a user's top-5 look like?

Same procedure as the SBERT notebook: pick an active user, show their recent positives, then show what Tag SVD recommends.

In [5]:
recipes = load_recipes()
user_positives = train[train['rating'] >= 4].groupby('user_id').size().sort_values(ascending=False)
demo_user = int(user_positives.index[100])
print(f"Demo user: {demo_user}  ({user_positives[demo_user]} positives in train)")

user_train = train[(train['user_id'] == demo_user) & (train['rating'] >= 4)]
user_train = user_train.sort_values('date', ascending=False).head(5)
name_lookup = recipes.set_index(recipes['id'].astype(int))['name']

print("\nTheir 5 most recent 4+ star recipes:")
for _, row in user_train.iterrows():
    print(f"  {row['recipe_id']:>10}  {name_lookup.get(int(row['recipe_id']), '(unknown)')}")

recs = m_part.recommend(demo_user, k=5, exclude_seen=True)
print("\nTag SVD's top-5 recommendations:")
for rid in recs:
    print(f"  {rid:>10}  {name_lookup.get(rid, '(unknown)')}")

Demo user: 52282  (765 positives in train)

Their 5 most recent 4+ star recipes:
      203834  mom s pork tenderloin
       62469  pork with a blue cheese apple and mustard sauce
      116269  dutch slavinken   1
       34919  brats with whiskey glazed onions
      408682  mexican ground beef pie

Tag SVD's top-5 recommendations:
      306855  baked eggplant and ricotta rolls
       85330  figs in mavrodaphne wine with manouri cheese
       24736  peachy pork chops
       75064  wimpies
      360255  birthday dinner meatloaf


## 5. Eval across all three tracks

Same `evaluate()` harness as every other Stage 1 model.

In [6]:
results = []
with timer("validation eval"):
    r_val = evaluate(m_full, track='validation', k_values=(5, 10, 20, 100), seed=42)
with timer("warm eval"):
    r_warm = evaluate(m_part, track='warm', k_values=(5, 10, 20, 100), n_users=2000, seed=42)
with timer("cold eval"):
    r_cold = evaluate(m_full, track='cold', k_values=(5, 10, 20, 100), seed=42)

for track, res in [('validation', r_val), ('warm', r_warm), ('cold', r_cold)]:
    results.append({
        'track':         track,
        'recall@5':      res['recall@5']   * 100,
        'recall@10':     res['recall@10']  * 100,
        'recall@20':     res['recall@20']  * 100,
        'recall@100':    res['recall@100'] * 100,
        'ndcg@10':       res['ndcg@10']    * 100,
        'mrr':           res['mrr']        * 100,
        'n_users':       res['n_users_evaluated'],
    })

pd.DataFrame(results).set_index('track').round(4)

  [32.2s] validation eval


  [11.2s] warm eval


  [58.2s] cold eval


,recall@5,recall@10,recall@20,recall@100,ndcg@10,mrr,n_users
track,,,,,,,
validation,0.0169,0.0169,0.0339,0.2881,0.0085,0.0122,5900
warm,0.0000,0.0000,0.0000,0.5000,0.0000,0.0128,2000
cold,0.0096,0.0096,0.0385,0.1636,0.0037,0.0064,10393


## 6. Comparison vs SBERT + Popularity

Tag SVD's numbers in context. SBERT and Popularity numbers below come from `sentence_bert_smoke.ipynb` (same seed, same eval protocol).

In [7]:
# Stage 1 leaderboard — Recall@10 (isolated comparison) AND Recall@100 (pipeline-relevant
# candidate-pool coverage). Reference numbers come from the joint sweep saved at
# data/processed/stage1_leaderboard.csv. The Tag SVD row is filled live from this notebook's
# eval cell (cell 12) — if you re-run that, the row below auto-updates.

reference = pd.DataFrame([
    {'model': 'Popularity',              'val_r10': 0.000, 'val_r100': 0.000,
                                          'warm_r10': 2.950, 'warm_r100': 11.550,
                                          'cold_r10': 0.000, 'cold_r100': 0.000},
    {'model': 'Tag SVD (this notebook)',  'val_r10': r_val['recall@10']*100,  'val_r100': r_val['recall@100']*100 if 'recall@100' in r_val else None,
                                          'warm_r10': r_warm['recall@10']*100, 'warm_r100': r_warm['recall@100']*100 if 'recall@100' in r_warm else None,
                                          'cold_r10': r_cold['recall@10']*100, 'cold_r100': r_cold['recall@100']*100 if 'recall@100' in r_cold else None},
    {'model': 'SBERT content',            'val_r10': 0.169, 'val_r100': 0.373,
                                          'warm_r10': 0.150, 'warm_r100': 0.700,
                                          'cold_r10': 0.087, 'cold_r100': 0.452},
    {'model': 'SBERT + Tag SVD (w=0.25)', 'val_r10': 0.102, 'val_r100': 0.458,
                                          'warm_r10': 0.100, 'warm_r100': 0.800,
                                          'cold_r10': 0.087, 'cold_r100': 0.366},
]).set_index('model').round(3)
reference

,val_r10,val_r100,warm_r10,warm_r100,cold_r10,cold_r100
model,,,,,,
Popularity,0.000,0.000,2.95,11.55,0.000,0.000
Tag SVD (this notebook),0.017,0.288,0.00,0.50,0.010,0.164
SBERT content,0.169,0.373,0.15,0.70,0.087,0.452
SBERT + Tag SVD (w=0.25),0.102,0.458,0.10,0.80,0.087,0.366


## 7. Summary

**Headline (at @10)**: Tag SVD content significantly **underperforms** SBERT as a standalone Stage 1 model.

| | Tag SVD | SBERT | Gap |
|---|---|---|---|
| Validation Recall@10 | 0.017% | 0.169% | SBERT ~10× |
| Cold Recall@10 | 0.010% | 0.087% | SBERT ~9× |
| Warm Recall@10 | 0.000% | 0.150% | both content lose on warm (expected) |

**Headline (at @100 — pipeline-relevant)**: Tag SVD remains the weakest standalone model, but its features become **useful when concatenated** with SBERT.

| | Tag SVD | SBERT | SBERT+TagSVD (w=0.25) |
|---|---|---|---|
| Validation Recall@100 | 0.288% | 0.373% | **0.458%** |
| Warm Recall@100 | 0.500% | 0.700% | **0.800%** |
| Cold Recall@100 | 0.164% | **0.452%** | 0.366% |

This is the most interesting finding for Tag SVD: alone it's a weak ranker, but as an *additional feature space* in a richer recipe representation it contributes complementary candidates to SBERT's top-100. See Iteration 3 in `sentence_bert_smoke.ipynb` for the full sweep.

**Why Tag SVD underperforms standalone** (informed guess):
- Tags are *coarse* — labels like `vegetarian`, `30-minutes-or-less`, `course` group thousands of recipes together. Cosine similarity in tag-SVD space treats all "vegetarian appetizers" as nearly identical.
- 100 SVD dims for ~140 selected tags is essentially a slightly-compressed one-hot lookup.
- 7 nutrition dims further bucket recipes by macros — same kcal + protein = high similarity, regardless of dish identity.
- No discrimination on specific ingredients (`salmon` vs `tofu`) — those live in the free-form text, not the tags.

**Sanity verified**: at `content_weight=0.0` the model falls back to popularity (Warm Recall@10 = 2.95%, matching the global popularity baseline exactly). The 0% warm number at default `content_weight=1.0` is a real signal-quality issue, not a wiring bug.

**Implication for the project narrative**:
- Free-form recipe text (SBERT) carries meaningfully more cold-start signal than structured tags + nutrition at @10.
- At @100 (Stage 2 candidate-pool scale), Tag SVD features become a useful *augmentation* to SBERT — they add variety in the top-100 that improves coverage on val/warm.
- The right Stage 1 choice depends on which K you optimize for. For our pipeline (Stage 1 produces top-100, Stage 2 re-ranks), @100 is the more honest comparison.

**Next**: build EASE (CF model) for warm-track wins. Tag SVD is documented as a Stage 1 baseline; we won't iterate on it further as a standalone model.